# LineScout · Milestone 2 GPU ingestion pipeline

Turn a folder of raw source artwork into a **validated LineScout gallery** plus
**feature shards**, on a free Colab GPU — no account, no paid tier, nothing to
install locally.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junosapollo/drawable/blob/main/ml/colab/linescout_gpu_pipeline.ipynb)

**What you get at the end**

```
<DRIVE_ROOT>/gallery/<DATASET_VERSION>/
    originals/<asset_id>.<ext>      source image, copied byte-for-byte
    line_art/<asset_id>.png         extracted (or native) line art
    thumbnails/<asset_id>.png       reference-panel tile
    manifest.json                   validated against ml/linescout_ml/manifest.py
    _pipeline/candidates.jsonl      resumable per-image state
    _pipeline/run_report.json       config, GPU, environment, checkpoint pins, counts
<DRIVE_ROOT>/indexes/<DATASET_VERSION>/
    mobileclip2_s2/{index.json, shard-*.npz}
    dinov2_vits14/{index.json, shard-*.npz}
```

**How it is built.** This notebook is a thin driver: every stage lives in the
repository's `linescout_ml.colab` package, which is typed, linted, and unit
tested in CI. The cells below configure that package, call it, and show you what
came out. When a stage misbehaves, the fix belongs in a reviewed module, not in a
notebook cell.

**Reproducible by construction.** A dataset is only worth rebuilding if you can
name what produced it, so three things are pinned *and recorded* rather than
assumed: the **code** (one immutable commit, HEAD verified before anything runs),
the **environment** (`ml/colab/requirements-colab.txt`, plus a snapshot of what
Colab's runtime already had installed), and the **weights**
(`ml/linescout_ml/colab/models.lock.json`: repository revision, file, SHA-256,
verified before a model loads). All three land in `run_report.json` and in the
manifest's `provenance` block.

| # | Stage | GPU | What it writes |
|---|---|---|---|
| 1 | Configure | — | paths, sources, toggles |
| 2 | Environment | — | Drive mount, **pinned checkout**, **pinned packages** |
| 2b–2d | Resolve, freeze, install | no | what to install, then what actually ran |
| 3 | Sanity check | no | full run on the committed fixture |
| 5 | Discover | no | candidates, asset ids, splits |
| 6 | **Extract line art** | **yes** | originals, line art, thumbnails |
| 7 | Measure | no | ink, text, quality, pHash, crop |
| 8 | De-duplicate | no | near-duplicate groups dropped |
| 9 | **Label** | **yes** | provisional style/scope, SFW gate |
| 10 | **Embed** | **yes** | MobileCLIP2 + DINOv2 shards |
| 11 | Build manifest | no | `manifest.json`, validated |
| 12 | Export | no | zip, Drive copy, run report |

> **Budget.** On a T4, extraction runs at roughly 1-2 images/second at 1024 px
> and embedding at ~50 images/second, so allow about 25 minutes per 1,000 images
> for the whole pipeline. Every stage persists before the next one starts, so a
> disconnect costs you the in-flight batch, not the run.

> **Licences.** The notebook downloads public model weights and writes licence
> identifiers into a provenance manifest. It refuses to run with a placeholder
> licence, and section 15 spells out what each model's terms actually are —
> including the fact that **MobileCLIP2 weights are research-only**. Read that
> before redistributing anything this pipeline produces.

## 0 · Before you start

1. **Runtime → Change runtime type → T4 GPU.** Cell 2 prints what you actually
   got. A CPU runtime still works end to end; extraction is simply ~30x slower.
2. **Do not edit `REPO_PIN` casually.** Cell 1 names one immutable commit of
   `junosapollo/drawable` and cell 2 refuses to run unless HEAD matches it, so
   "re-run this six months from now" has an answer. Point it at your own SHA when
   you need a different tree — and read the diff first.
3. **Upload your sources to Drive**, one folder per dataset:
   ```
   MyDrive/LineScout/sources/
       amateur_drawings/…
       manga109/<title>/page-001.png
       met_openaccess/…
   ```
   Folders are scanned recursively. `.png .jpg .jpeg .webp .bmp .gif` are picked
   up; everything else is ignored.
4. **Know your licence.** Each source needs a `license_id` you verified yourself.
   Several datasets this project targets (Manga109, eBDtheque, Human-Art) require
   an access application and forbid redistribution — start those early and record
   exactly what you were granted.
5. **No tokens, no secrets.** Weights come from public Hugging Face repos and
   `dl.fbaipublicfiles.com`; the repository clone is public. A mirror cannot
   smuggle in different code: whatever arrives is checked against `REPO_PIN`.

## 1 · Configuration

The only cell you normally need to edit, and the only place the three pins are
configured: which commit the code comes from, which checkpoint policy applies, and
where the package pins live.

In [ ]:
# @title 1 · Run configuration (edit me — Colab renders this cell as a form)
from pathlib import Path

# --- which code ----------------------------------------------------------------
# One immutable commit, not "main": a dataset rebuilt next year has to be
# attributable to the tree that produced it. Cell 2 clones this SHA and refuses to
# continue unless HEAD equals it. ml/colab/README.md names the current pin.
REPO_URL = "https://github.com/junosapollo/drawable.git"  # @param {type:"string"}
REPO_PIN = "06ae97663c8c322f5020ae3574cb9ec55f00dbe3"  # @param {type:"string"}
# Optional read-only fallback, tried only if REPO_URL is unreachable. It cannot
# change what runs: whatever arrives is verified against REPO_PIN.
REPO_MIRROR_URL = "https://github.com/Sehaan-1/drawable.git"  # @param {type:"string"}
REPO_ALLOW_MIRROR = True  # @param {type:"boolean"}
# Blank clones into /content/drawable. Point it at a checkout you already have
# (e.g. /content/drive/MyDrive/drawable) to skip the download: it is verified the
# same way, fetched onto the pin if it is behind, and never reset under your feet.
REPO_DIR = ""  # @param {type:"string"}
# True reuses a checkout with uncommitted edits and records the run as dirty.
# False refuses it, because nothing you have not committed can be re-created.
REPO_ALLOW_DIRTY = True  # @param {type:"boolean"}

# --- which model weights -------------------------------------------------------
# ml/linescout_ml/colab/models.lock.json pins every checkpoint's repository
# revision, file, and SHA-256. "strict" refuses to load anything unpinned;
# "record" verifies each pinned digest and records the ones it computes itself;
# "off" is for debugging on a machine you already trust.
CHECKPOINT_POLICY = "record"  # @param ["strict", "record", "off"]
CHECKPOINT_CACHE_DIR = "/root/.cache/linescout/checkpoints"  # @param {type:"string"}
# Optional: pin the bytes of that lock file too. Blank trusts REPO_PIN, which
# already fixes them.
MODEL_LOCK_SHA256 = ""  # @param {type:"string"}

# --- which packages ------------------------------------------------------------
AUTO_INSTALL = True  # @param {type:"boolean"}  # False prints the plan, installs nothing

# --- the run itself ------------------------------------------------------------
DATASET_VERSION = "2026.09.06-colab1"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/LineScout"  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}

# One entry per raw dataset. `preset` keys are printed by cell 2b; `license_id` is
# mandatory and must be something you verified — the pipeline refuses to write a
# provenance manifest with a placeholder in it.
SOURCES = [
    {
        "preset": "amateur_drawings",
        "folder": "amateur_drawings",  # relative to SOURCES_ROOT, or an absolute "root"
        "license_id": "REPLACE-ME",
        "extractor": "",  # optional: anime2sketch | informative_drawings
        "sfw_method": "",  # optional: source_rating | opennsfw2 | source_rating+opennsfw2
        #   | manual — your own verdict, recorded as method="manual", never a classifier's
        #   Blank (usual) = the preset's choice, which for a scraped source is the gate
        "work_grouping": "",  # optional: filename | parent_dir
    },
]

LIMIT_PER_SOURCE = 0  # @param {type:"integer"}  # 0 = no limit; try 20 for a first smoke run
LINE_ART_RESOLUTION = 1024  # @param {type:"slider", min:256, max:2048, step:64}
THUMBNAIL_SIZE = 256  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"slider", min:1, max:32, step:1}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
RUN_EXTRACTION = True  # @param {type:"boolean"}
RUN_DEDUPE = True  # @param {type:"boolean"}
RUN_LABELS = True  # @param {type:"boolean"}
RUN_EMBEDDINGS = True  # @param {type:"boolean"}
EMBEDDERS = "mobileclip2_s2,dinov2_vits14"  # @param {type:"string"}
LABELER_MODEL = "MobileCLIP2-S2"  # @param {type:"string"}
EXPORT_TO_DRIVE = True  # @param {type:"boolean"}
DOWNLOAD_ZIP = True  # @param {type:"boolean"}

SOURCES_ROOT = Path(DRIVE_ROOT) / "sources"
GALLERY_ROOT = Path(DRIVE_ROOT) / "gallery" / DATASET_VERSION
INDEX_ROOT = Path(DRIVE_ROOT) / "indexes" / DATASET_VERSION
ZIP_PATH = Path("/content") / f"linescout-{DATASET_VERSION}.zip"

print(f"dataset version : {DATASET_VERSION}")
print(f"sources         : {SOURCES_ROOT}")
print(f"gallery         : {GALLERY_ROOT}")
print(f"indexes         : {INDEX_ROOT}")
print(f"sources declared: {len(SOURCES)}")

## 2 · Environment

Four cells, in an order that is deliberate.

* **2 · the code.** The GPU is probed, Drive is mounted, and the repository is
  placed at `REPO_PIN` — cloned shallowly, or reused from a checkout you point at.
  Either way the verdict is the package's own `find_or_checkout`, which compares
  HEAD against the pin, fetches a stale checkout onto it, and never resets a
  working tree you have edited.
* **2b · what this run needs.** Presets expand into typed `SourceSpec` objects and
  licences are checked *before* a single package installs, because the install plan
  is a function of the answer.
* **2c · torch.** Colab's preinstalled CUDA build is frozen into a constraints file
  that every later install honours. Nothing installs torch.
* **2d · the rest.** Packages go in at the versions
  `ml/colab/requirements-colab.txt` pins, and the runtime that results is
  *recorded* and diffed against a baseline — a T4 image ships its own numpy, torch,
  CUDA, and TensorFlow, and pretending otherwise is how a "reproducible" run turns
  out to have executed on software nobody looked at.

In [ ]:
# @title 2 · GPU, Drive, and the pinned checkout
"""Prove the hardware, then get the code to one commit — before anything installs.

The verdict belongs to `find_or_checkout`, the same function CI exercises. The
`git clone` below exists only because a runtime with nothing installed cannot
import the code that tells it how to check itself out: it is shallow, disposable,
and immediately handed to the package, which moves the directory onto `REPO_PIN`
and refuses to continue unless HEAD equals it."""
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def sh(command: list[str], cwd: Path | None = None) -> int:
    """Run a command, echoing it, and hand back the exit code."""
    print("$ " + " ".join(command))
    return subprocess.run(command, cwd=cwd, check=False).returncode


def is_checkout(path: Path) -> bool:
    return (path / "ml" / "linescout_ml").is_dir()


probe = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(probe.stdout.strip() or "nvidia-smi not found — this is a CPU runtime")

if MOUNT_DRIVE and importlib.util.find_spec("google.colab") is not None:
    from google.colab import drive

    drive.mount("/content/drive")
elif MOUNT_DRIVE:
    print("not running in Colab; skipping the Drive mount")

CANDIDATES = [
    Path(value)
    for value in (
        REPO_DIR,
        os.environ.get("LINESCOUT_REPO", ""),
        os.environ.get("REPO_DIR", ""),
        str(Path(DRIVE_ROOT).parent / "drawable"),
    )
    if value
]
if REPO_DIR and not is_checkout(Path(REPO_DIR)):
    raise SystemExit(
        f"REPO_DIR={REPO_DIR} is not a LineScout checkout (no ml/linescout_ml in it). "
        "Leave it blank to clone the pinned revision, or fix the path."
    )

CLONE_TARGET = Path("/content/drawable")
BOOTSTRAP = next((path for path in CANDIDATES if is_checkout(path)), CLONE_TARGET)
if not is_checkout(BOOTSTRAP) and sh(["git", "clone", "--depth", "1", REPO_URL, str(BOOTSTRAP)]):
    raise SystemExit(f"git clone of {REPO_URL} failed; set REPO_DIR to a checkout you trust")

sys.path.insert(0, str(BOOTSTRAP / "ml"))

from linescout_ml.colab import (  # noqa: E402
    CheckoutError,
    RevisionMismatchError,
    find_or_checkout,
    pin_is_finalised,
)

try:
    CHECKOUT = find_or_checkout(
        revision=REPO_PIN,
        url=REPO_URL,
        mirrors=(REPO_MIRROR_URL,) if REPO_ALLOW_MIRROR else (),
        preferred=[*CANDIDATES, BOOTSTRAP],
        target=CLONE_TARGET,
        allow_dirty=REPO_ALLOW_DIRTY,
        on_note=lambda message: print(f"  note: {message}"),
    )
except (CheckoutError, RevisionMismatchError) as error:
    raise SystemExit(f"the pinned checkout is not usable: {error}") from None

REPO = Path(CHECKOUT.path)
print()
for line in CHECKOUT.describe():
    print(line)
if not pin_is_finalised(CHECKOUT.requested_revision):
    print(f"  note: {CHECKOUT.requested_revision} is this repository's placeholder pin")

## 2b · Resolve what this run is

`SOURCES` is form data: a preset per dataset, optional overrides, and a licence
each source has to earn its way into. This cell turns it into `SourceSpec` objects
and stops the run there if a licence is a placeholder, a preset is unknown, or a
combination is impossible (native line art cannot also name an extractor).

Doing it now — two cells before the install — is what lets cell 2d size the
environment from the answer instead of from a hard-coded package list.

In [ ]:
# @title 2b · Resolve the sources, then say what they need
"""Licensing and preset expansion happen before any install, for a practical
reason: *what to install* is a function of *what these entries turn out to be*.

The package owns both halves, so the Human-Art preset asks for the NSFW gate
whether or not you wrote `"sfw_method": "opennsfw2"` yourself — and the notebook
never re-implements preset logic in a language CI does not lint.
"""
from linescout_ml.colab import SourceConfigurationError, preset_table, resolve_sources

print(f"{'preset':<24} licence you must verify")
for row in preset_table():
    print(f"{row['key']:<24} {row['license_note']}")
    print(f"{'':<24} ↳ {row['description']}")

try:
    SOURCE_SPECS = resolve_sources(SOURCES, sources_root=SOURCES_ROOT)
except SourceConfigurationError as error:
    raise SystemExit(
        "source configuration is not usable — nothing was installed yet:\n" + str(error)
    ) from None

print()
for spec in SOURCE_SPECS:
    present = "ok" if spec.root.is_dir() else "MISSING"
    print(
        f"  [{present:>7}] {spec.name:<20} origin={spec.origin.value:<19}"
        f" extractor={spec.extractor:<20} sfw={spec.sfw_method:<24} licence={spec.license_id}"
    )
    print(f"            {spec.root}")
    if spec.uses_extractor:
        print("            needs a GPU extractor stage (controlnet-aux)")
    if spec.requires_nsfw:
        print("            needs the opennsfw2 gate (its Keras backend comes from the runtime)")
print(f"\n{len(SOURCE_SPECS)} source(s) resolved; cell 2d turns them into an install plan")

## 2c · Freeze the runtime's torch

The one cell that exists because of how Colab is put together: pip must not be
allowed to touch `torch`, and a stray `torch.py` must not be allowed to shadow it.
It also defines the `pip()` helper the rest of the notebook uses, so every install
honours the same constraints file.

In [ ]:
# @title 2c · Freeze the torch this runtime already has
"""Two Colab-specific hazards, handled once.

A stray `torch.py` in the working directory shadows the installed package, and a
plain `pip install` of anything *else* is free to "fix" a version conflict by
replacing Colab's CUDA build of torch with a CPU wheel. So the runtime's own torch
and torchvision are written into a constraints file that every install in this
notebook passes to pip. Nothing here installs torch."""
import glob
import os
import subprocess
import sys
from pathlib import Path

from linescout_ml.colab import runtime_constraints

for name in ("torch", "torchvision", "transformers", "xformers"):
    for shadow in glob.glob(f"{name}.py"):
        os.remove(shadow)
        print(f"removed {shadow} — it would shadow the installed package")

CONSTRAINTS = Path("/tmp/linescout-constraints.txt")
PINNED = runtime_constraints(("torch", "torchvision"), write_to=CONSTRAINTS)
print("constraints:", ", ".join(PINNED) or "no torch importable here (CPU runtime)")


def pip(*specs: str) -> None:
    """Install pinned specs under the constraints file, and fail loudly."""
    command = [sys.executable, "-m", "pip", "install", "--quiet", *specs]
    if CONSTRAINTS.is_file():
        command += ["-c", str(CONSTRAINTS)]
    print("$ " + " ".join(command))
    if subprocess.run(command, check=False).returncode:
        raise SystemExit(
            f"pip refused {' '.join(specs)}: the pinned environment does not fit this runtime. "
            "Say so in the run report instead of installing unpinned."
        )

## 2d · Install the pinned packages, then record the runtime

Two halves in one cell, in this order: **plan → install** from the environment
spec, then **observe → report** what the runtime actually ended up with.
`REPRO_FLAGS` is what cell 4 stamps into the config, and
`/content/linescout-runtime.json` is the machine-readable version of the same
facts.

In [ ]:
# @title 2d · Install the pinned subset, then record what actually ran
"""The plan comes from cell 2b; the versions come from
`ml/colab/requirements-colab.txt`, one pin per line, the same pins `ml/uv.lock`
resolves for CI.

Recording is the other half of the job. A T4 image arrives with numpy, torch, CUDA,
and TensorFlow already installed, and none of it is a dockerfile you own, so the
honest artifact is a snapshot of what the runtime *turned out* to be — diffed
against `ml/colab/runtime-baseline.json`, with the differences printed rather than
smoothed over."""
import json
from pathlib import Path

from linescout_ml.colab import (
    compare_runtime,
    load_baseline,
    load_model_lock,
    load_requirement_spec,
    lock_digest,
    plan_environment,
    runtime_report,
)

SPEC = load_requirement_spec(REPO / "ml" / "colab" / "requirements-colab.txt")
if SPEC is None:
    raise SystemExit(f"no pinned environment spec under {REPO / 'ml/colab'}")

PLAN = plan_environment(
    SOURCE_SPECS,
    spec=SPEC,
    extract_line_art=RUN_EXTRACTION,
    label=RUN_LABELS,
    embed=RUN_EMBEDDINGS,
    embedders=[key.strip() for key in EMBEDDERS.split(",") if key.strip()],
)
print()
for line in PLAN.describe():
    print(line)

if not AUTO_INSTALL:
    print("AUTO_INSTALL is off — nothing installed; expect a MissingDependencyError later")
elif PLAN.pip_specs:
    pip(*PLAN.pip_specs)
else:
    print("every package this run needs is already installed at the pinned version")

MODEL_LOCK = REPO / "ml" / "linescout_ml" / "colab" / "models.lock.json"
MODEL_LOCK_SHA = lock_digest(MODEL_LOCK)["sha256"]
if MODEL_LOCK_SHA256.strip() and MODEL_LOCK_SHA256.strip().lower() != MODEL_LOCK_SHA:
    raise SystemExit(
        f"MODEL_LOCK_SHA256 says {MODEL_LOCK_SHA256.strip()[:12]} but {MODEL_LOCK.name} hashes "
        f"to {MODEL_LOCK_SHA[:12]} — these are not the checkpoint pins you meant to run"
    )
LOCK = load_model_lock(MODEL_LOCK)
UNPINNED = [artifact.id for artifact in LOCK.unpinned()]

RUNTIME = runtime_report()
BASELINE = load_baseline(REPO / "ml" / "colab" / "runtime-baseline.json")
DRIFT = compare_runtime(RUNTIME, BASELINE)
RUNTIME_SNAPSHOT = Path("/content/linescout-runtime.json")
RUNTIME_SNAPSHOT.write_text(json.dumps(RUNTIME, indent=2, sort_keys=True) + "\n")

print()
print(f"python        {RUNTIME['python']} on {RUNTIME['runtime']}")
print(f"torch         {RUNTIME['packages'].get('torch') or 'absent'}")
print(f"environment   {(SPEC.sha256 or 'unhashed')[:12]}  ({Path(SPEC.path).name})")
print(
    f"checkpoints   {(MODEL_LOCK_SHA or 'unhashed')[:12]}  ({len(LOCK.artifacts)} artifacts, "
    f"{len(UNPINNED)} without a pinned digest, policy {CHECKPOINT_POLICY})"
)
if BASELINE is None:
    print("baseline      none recorded in this checkout")
elif DRIFT:
    print("baseline      differs from ml/colab/runtime-baseline.json:")
    for name, delta in sorted(DRIFT.items()):
        print(f"                {name}: recorded {delta['expected']} → here {delta['actual']}")
else:
    print("baseline      matches ml/colab/runtime-baseline.json")
print(f"runtime json  {RUNTIME_SNAPSHOT} (cell 12 copies it next to the manifest)")

REPRO_FLAGS = {
    "source_repo": CHECKOUT.repo,
    "source_revision": CHECKOUT.revision,
    "source_dirty": CHECKOUT.dirty,
    "source_action": CHECKOUT.action,
    "environment_sha256": SPEC.sha256,
    "checkpoint_policy": CHECKPOINT_POLICY,
    "checkpoint_cache_dir": Path(CHECKPOINT_CACHE_DIR) if CHECKPOINT_CACHE_DIR else None,
    "model_lock_path": MODEL_LOCK,
}

## 3 · Sanity check (no GPU, ~10 seconds)

Runs the **whole pipeline** against the synthetic fixture committed to the
repository — the same 24 generated drawings the API serves in fixture mode. No
detector, no CLIP, no embeddings. It proves the plumbing (discovery → normalised
line art → measurements → de-duplication → manifest → zip) works in *this*
runtime before you spend GPU minutes on real data.

The fixture deliberately contains repeated pages, so de-duplication keeps about
10 of the 24 candidates. That is the stage working, not data loss.

Skip this cell once you have seen it pass.

In [ ]:
# @title 3 · Dry run on the committed synthetic fixture
from linescout_ml.colab import PipelineConfig, PipelineRunner, preset_source

FIXTURE = REPO / "ml" / "fixtures" / "synthetic" / "originals"
DRY_ROOT = Path("/content/linescout-dryrun")

dry_config = PipelineConfig(
    dataset_version=DATASET_VERSION,
    output_root=DRY_ROOT / "gallery",
    embeddings_root=DRY_ROOT / "indexes",
    sources=[preset_source("synthetic", root=FIXTURE, license_id="synthetic-fixture")],
    label=False,
    embed=False,
    device="cpu",
    line_art_resolution=512,
    thumbnail_size=THUMBNAIL_SIZE,
)
dry_report = PipelineRunner(dry_config, progress=lambda stage, done, total: None).run_all(
    zip_path=DRY_ROOT / "dryrun.zip"
)

counts = dry_report["summary"]["candidates"]
print(
    f"candidates : {counts['total']} found, {counts['active']} kept,"
    f" {counts['duplicates']} duplicates, {counts['skipped']} skipped"
)
print(f"records    : {dry_report['summary']['total']}")
print(
    f"zip        : {dry_report['outputs']['zip']['path']}"
    f" ({dry_report['outputs']['zip']['bytes']} bytes)"
)
print(f"sha256     : {dry_report['outputs']['zip']['sha256'][:16]}…")
print("stages     :", ", ".join(stage["name"] for stage in dry_report["stages"]))

## 4 · Build the run configuration

Everything was established upstream: `SOURCE_SPECS` came from 2b, the verified
checkout from 2, the recorded environment and the checkpoint digests from 2d.
This cell only assembles them into the one typed `PipelineConfig` the runner
consumes, so what a run *recorded* and what a run *did* cannot disagree.

Re-run it after changing cell 1; it is cheap, and every stage cell reads the
result.

In [ ]:
# @title 4 · Pipeline configuration
"""Assembled from what the earlier cells established, never re-derived from the
form: the sources are already resolved, the checkout is already verified, the
environment and checkpoint digests are already recorded."""
from linescout_ml.colab import COLAB_REPO, PipelineConfig

CONFIG = PipelineConfig(
    dataset_version=DATASET_VERSION,
    output_root=GALLERY_ROOT,
    embeddings_root=INDEX_ROOT,
    sources=SOURCE_SPECS,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    limit_per_source=LIMIT_PER_SOURCE or None,
    thumbnail_size=THUMBNAIL_SIZE,
    line_art_resolution=LINE_ART_RESOLUTION,
    extract_line_art=RUN_EXTRACTION,
    dedupe=RUN_DEDUPE,
    label=RUN_LABELS,
    embed=RUN_EMBEDDINGS,
    embedders=[key.strip() for key in EMBEDDERS.split(",") if key.strip()],
    labeler_model=LABELER_MODEL,
    # What this run is attributable to. Recorded, not asserted: each value came
    # from measuring this runtime, and both the manifest and the run report carry
    # it, so a later rebuild can be compared instead of argued about.
    source_repo=CHECKOUT.repo or COLAB_REPO,
    source_revision=CHECKOUT.revision,
    source_dirty=CHECKOUT.dirty,
    source_action=CHECKOUT.action,
    environment_sha256=SPEC.sha256,
    model_lock_sha256=MODEL_LOCK_SHA,
    checkpoint_policy=CHECKPOINT_POLICY,
    checkpoint_cache_dir=Path(CHECKPOINT_CACHE_DIR) if CHECKPOINT_CACHE_DIR else None,
    model_lock_path=MODEL_LOCK,
)

print(f"\n{len(CONFIG.sources)} source(s) -> {CONFIG.output_root}")
for spec in CONFIG.sources:
    print(f"  {spec.name:<20} {spec.origin.value:<19} {spec.license_id}")
print(
    f"code        {(CONFIG.source_revision or '')[:12]}"
    f" ({'DIRTY' if CONFIG.source_dirty else 'clean'}, via {CONFIG.source_action})"
)
print(
    f"environment {(CONFIG.environment_sha256 or 'unhashed')[:12]}"
    f"   checkpoints {(CONFIG.model_lock_sha256 or 'unhashed')[:12]}"
    f" ({CONFIG.checkpoint_policy})"
)

In [ ]:
# @title 4b · Runner: progress bars and live notes
from tqdm.auto import tqdm

from linescout_ml.colab import PipelineRunner

_bars: dict[str, tqdm] = {}


def _progress(stage: str, done: int, total: int) -> None:
    bar = _bars.get(stage)
    if bar is None or bar.total != total:
        if bar is not None:
            bar.close()
        bar = tqdm(total=total, desc=stage, unit="img", leave=True)
        _bars[stage] = bar
    bar.n = min(done, total or 0)
    bar.refresh()


def _note(message: str) -> None:
    print(f"  note: {message}")


def close_bars() -> None:
    for bar in _bars.values():
        bar.close()
    _bars.clear()


RUNNER = PipelineRunner(CONFIG, progress=_progress, on_note=_note)
print(f"device : {RUNNER.device}")
print(f"gallery: {CONFIG.output_root}")
print(f"state  : {CONFIG.candidates_path}")
print(f"code   : {(CONFIG.source_revision or '')[:12]} from {CONFIG.source_repo}")
print(f"weights: policy {CONFIG.checkpoint_policy}, cache {RUNNER.checkpoints.cache_dir}")

## 5 · Discover

Walks each source folder, assigns the deterministic asset id
(`ls_<dataset>_<sha256 prefix>`), groups images into *source works*, and derives
each work's split from a hash. That is what makes the manifest rule "assets from
one source work never cross splits" true by construction rather than by post-hoc
checking.

State is written to `_pipeline/candidates.jsonl` after every stage, so a
restarted runtime resumes instead of starting over.

In [ ]:
# @title 5 · Discover sources
from collections import Counter

from linescout_ml.colab import candidate_summary

discover_stage = RUNNER.discover()
close_bars()

print(f"candidates : {discover_stage.processed}")
print(f"state file : {RUNNER.config.candidates_path}")
for note in discover_stage.notes:
    print(f"  note: {note}")
print(f"summary    : {candidate_summary(RUNNER.store)}")
print(f"splits     : {dict(Counter(c.split.value for c in RUNNER.store))}")

## 6 · Extract line art · *GPU*

Runs the extractor each source asked for — `anime2sketch` (Anime2Sketch weights)
for manga and anime, `informative_drawings` (Chan *et al.*) for photographs,
paintings, and academic drawing. Both are served by `controlnet_aux` from
Hugging Face, so there is no Google Drive checkpoint hunting. Sources marked
`native_line_art` are normalised (alpha flattened onto white, EXIF orientation
applied) and never re-drawn.

Three files per asset are written here: the original (copied byte-for-byte,
suffix preserved), the PNG line art, and the PNG thumbnail. `controlnet_aux` works on a multiple of 64, so the line art is resized
back to the source geometry — the API serves line art and thumbnail side by side,
and the crop overlay assumes they line up pixel for pixel.

If CUDA runs out of memory on one huge scan, the runner clears the cache and
retries that image at half resolution instead of ending the run.

In [ ]:
# @title 6 · Extract line art (GPU)
extract_stage = RUNNER.run_extract()
close_bars()

print(f"processed : {extract_stage.processed}")
print(f"skipped   : {extract_stage.skipped} (already on disk from an earlier run)")
print(f"failed    : {extract_stage.failed}")
print(f"seconds   : {extract_stage.seconds:.1f}")
for note in extract_stage.notes:
    print(f"  note: {note}")

In [ ]:
# @title 6b · Compare a few originals with their line art
import matplotlib.pyplot as plt
from PIL import Image

from linescout_ml.colab import asset_paths


def show_pairs(limit: int = 6) -> None:
    """Grid of original | line art for the first `limit` extracted candidates."""
    ready = [c for c in RUNNER.store.active if c.line_art_path][:limit]
    if not ready:
        print("nothing extracted yet")
        return
    root = RUNNER.config.output_root
    figure, axes = plt.subplots(len(ready), 2, figsize=(8, 4 * len(ready)), squeeze=False)
    for row, candidate in enumerate(ready):
        original = candidate.original_path or asset_paths(candidate.asset_id).original
        axes[row, 0].imshow(Image.open(root / original))
        axes[row, 0].set_title(
            f"original · {candidate.width}×{candidate.height} · {candidate.item_id[:32]}",
            fontsize=8,
        )
        axes[row, 1].imshow(Image.open(root / candidate.line_art_path), cmap="gray")
        axes[row, 1].set_title(
            f"{candidate.extraction_model or 'native'}@{candidate.extraction_version or '-'}",
            fontsize=8,
        )
        for axis in axes[row]:
            axis.axis("off")
    figure.tight_layout()
    plt.show()


show_pairs()

## 7 · Measure · *CPU*

Computes every number the manifest needs but no human should have to type, on a
view downscaled to 512 px so cost is flat regardless of source resolution:

* `ink_coverage` — pixels darker than 200/255, the same threshold the API's query
  preprocessing uses, so a stored asset and a query sketch are measured the same
  way
* `text_coverage` — share of the frame taken by glyph-like runs (speech bubbles,
  captions, watermarks, signatures); a connected-component heuristic, **not** OCR
* `quality_score` — a fixed weighted blend of resolution, ink density, paper
  cleanliness, and text contamination. It orders the curation queue; it does not
  judge art, and curation overrides it
* `phash` — 64-bit DCT perceptual hash, bit-for-bit compatible with
  `imagehash.phash`
* `crop` — the ink bounding box with 10 % padding, scaled back to source pixels,
  as the reviewer's starting frame

In [ ]:
# @title 7 · Measure line art
import numpy as np

measure_stage = RUNNER.run_measure()
close_bars()

print(f"processed : {measure_stage.processed}")
print(f"skipped   : {measure_stage.skipped}")
print(f"failed    : {measure_stage.failed}")
print(f"seconds   : {measure_stage.seconds:.1f}")
for note in measure_stage.notes:
    print(f"  note: {note}")

measured = [c.measurements for c in RUNNER.store.active if c.measurements]
if measured:
    for field in ("ink_coverage", "text_coverage", "quality_score"):
        values = np.array([getattr(item, field) for item in measured], dtype=float)
        print(
            f"{field:<15} min={values.min():.4f} median={np.median(values):.4f}"
            f" max={values.max():.4f}"
        )
    texty = [
        c for c in RUNNER.store.active if c.measurements and c.measurements.text_coverage > 0.02
    ]
    print(f"text > 2%   : {len(texty)} candidate(s) worth a look before curation")
    print(f"skipped     : {len([c for c in RUNNER.store if c.skip_reason])} candidate(s)")

## 8 · De-duplicate · *CPU*

Groups candidates whose pHash Hamming distance is within `dedupe_threshold`
(default 6) and keeps the lowest candidate key in each group. Duplicates are
marked in the state file, so they never reach the manifest, never get embedded,
and never appear in the export zip — but their files stay on disk, so you can
change your mind cheaply.

Marks are recomputed from scratch every time this cell runs. Too aggressive?

```python
RUNNER.config.dedupe_threshold = 3   # stricter, keeps more assets
RUNNER.config.dedupe = False         # or turn the stage off entirely
```

…then re-run this cell and everything you restore flows into the manifest again.
Scraped sources are full of rescaled re-uploads, so this is the cheapest quality
win in the pipeline — and it runs before you pay for labelling and embeddings.

In [ ]:
# @title 8 · De-duplicate by perceptual hash
from linescout_ml.colab import candidate_summary

dedupe_stage = RUNNER.run_dedupe()
close_bars()

print(f"duplicates removed : {dedupe_stage.processed}")
print(f"state              : {candidate_summary(RUNNER.store)}")
for note in dedupe_stage.notes:
    print(f"  note: {note}")

duplicates = [c for c in RUNNER.store if c.duplicate_of]
for candidate in duplicates[:10]:
    print(f"  {candidate.key}  ==  {candidate.duplicate_of}")
if len(duplicates) > 10:
    print(f"  … and {len(duplicates) - 10} more")

## 9 · Label · *GPU* (optional)

Asks the MobileCLIP2 text encoder to rank the five style families and the ten
scope buckets for each asset, using several prompts per label. The winner is
written as **provisional**: `labels.labelled_by` records whether a CLIP encoder
ranked it or the source default was used, the raw probabilities are stored
alongside, and the curation UI exists to correct all of it.

The SFW gate is separate and deliberately cheap, and the rule for what it trusts is
*who made the guarantee*: a publisher's own terms (museum open-access, an
application-gated research corpus) are recorded as `source_rating` with no classifier
run; anything scraped from a community pays for `opennsfw2`, including when the site
ships rating tags, because those tags are the claim under test. `+` runs both and keeps
the stricter verdict. The classifier screens the **original**, not the line art, because
extraction removes exactly the content a classifier needs to see. An asset that fails
the gate is written `quarantined` + `enabled=false`, which the manifest enforces as an
invariant.

`RUN_LABELS = False` in cell 1 turns off the CLIP ranking only: labels come from the
source defaults and `labels.labelled_by` says so. It does **not** skip the gate — a
source that needs `opennsfw2` still loads it, still costs GPU time, and still
quarantines. Screening is not a quality feature the *speed* toggle can trade away.

The one way to skip it is the honest one: `sfw_method="manual"` on a source, which
records `sfw.method="manual"` — your verdict, attributed to you, in every asset and in
the run report. Nothing pretends a classifier ran. An unreadable original fails
*closed* even then, and every asset arrives `review.state="unreviewed"` with
`enabled=false`, so a human still has to say yes before search serves it.

In [ ]:
# @title 9 · Zero-shot labels + SFW gate
from collections import Counter

label_stage = RUNNER.run_label()
close_bars()

print(f"processed : {label_stage.processed}")
print(f"skipped   : {label_stage.skipped}")
print(f"failed    : {label_stage.failed}")
for note in label_stage.notes:
    print(f"  note: {note}")

labelled = [c.labels for c in RUNNER.store.active if c.labels]
if labelled:
    print("style      :", dict(Counter(item.primary_style.value for item in labelled)))
    print("scope      :", dict(Counter(s.value for item in labelled for s in item.scopes)))
    print("labelled_by:", dict(Counter(item.labelled_by.value for item in labelled)))
    print(
        "sfw        :",
        dict(
            Counter(
                f"{item.sfw.method}:{'safe' if item.sfw.safe else 'UNSAFE'}" for item in labelled
            )
        ),
    )
    unsafe = [c for c in RUNNER.store.active if c.labels and not c.labels.sfw.safe]
    if unsafe:
        print(f"quarantined: {len(unsafe)} asset(s) will be written disabled")

## 10 · Embed · *GPU*

Writes L2-normalised feature vectors into resumable `.npz` shards:

* **MobileCLIP2-S2** (512-d) — text-aligned, and the same encoder that produced
  the labels, so the model is loaded once
* **DINOv2 ViT-S/14** (384-d) — self-supervised shape features. Line art has no
  colour and no texture, which is exactly where a text-aligned encoder is
  weakest, so the two are complementary; Milestone 4 concatenates them

`index.json` beside the shards records the model card, dimension, and licence of
whatever produced them, so an index can be rebuilt later without guessing which
checkpoint was in use. Re-running the cell embeds only the assets that are
missing — which is what makes a disconnect survivable.

In [ ]:
# @title 10 · Feature shards
import json

embed_stage = RUNNER.run_embed()
close_bars()

print(f"embedded : {embed_stage.processed}")
print(f"skipped  : {embed_stage.skipped} (already in the index)")
print(f"failed   : {embed_stage.failed}")
for note in embed_stage.notes:
    print(f"  note: {note}")

for key in CONFIG.embedders:
    index_path = CONFIG.resolved_embeddings_root() / key / "index.json"
    if not index_path.is_file():
        print(f"\n{key}: no index written")
        continue
    index = json.loads(index_path.read_text())
    print(
        f"\n{key}: {index['count']} vectors x {index['spec']['dim']}d"
        f" in {len(index['shards'])} shard(s)"
    )
    print(f"  model   : {index['spec']['name']} ({index['spec']['upstream']})")
    print(f"  licence : {index['spec']['license']}")

## 11 · Build the manifest · *CPU*

Assembles one `ManifestRecord` per surviving candidate and validates the whole
collection — including the "one source work, one split" rule — before writing
`manifest.json`. If a previous manifest exists it is **merged**, not replaced:
incoming records win on `asset_id` and existing records are preserved, so you can
add a second dataset next week without rebuilding the first.

The build refuses to write a manifest whose enabled assets are missing files,
because that is precisely what `services/api` would refuse to start against.

In [ ]:
# @title 11 · Manifest slice
from linescout_ml.colab import missing_files, summarise

MANIFEST = RUNNER.run_build()
close_bars()

summary = summarise(MANIFEST.records)
print(f"records        : {len(MANIFEST.records)}")
print(f"enabled        : {len(MANIFEST.enabled_records)}")
print(f"dataset_version: {MANIFEST.dataset_version}")
print(f"content_hash   : {MANIFEST.content_hash()[:16]}…")
print(f"missing files  : {len(missing_files(MANIFEST, CONFIG.output_root))}")
print(f"\nby style : {summary['by_style']}")
print(f"by scope : {summary['by_scope']}")
print(f"by split : {summary['by_split']}")
print(f"by origin: {summary['by_origin']}")
print(f"review   : {summary['by_review_state']}")
print(
    f"quality  : min={summary['quality_min']} median={summary['quality_median']}"
    f" max={summary['quality_max']}"
)

In [ ]:
# @title 11b · Validate it the way the project CLI does
from linescout_ml.cli import main as linescout_manifest_cli

exit_code = linescout_manifest_cli(["validate", str(CONFIG.manifest_path), "--require-files"])
print("exit code:", exit_code)
assert exit_code == 0, "the gallery manifest failed validation — do not export it"

## 12 · Export

Zips exactly the files the manifest vouches for (plus the manifest and the run
report), optionally mirrors the gallery to Drive, and starts a browser download.

Prefer the zip over the Drive copy for anything large: a gallery is tens of
thousands of small files, and Drive's FUSE mount is slow and prone to partial
writes. The zip's SHA-256 is recorded in the run report, so you can verify it
after the download.

In [ ]:
# @title 12 · Zip, Drive copy, run report
import shutil

outputs = RUNNER.run_export(
    zip_path=ZIP_PATH if DOWNLOAD_ZIP or EXPORT_TO_DRIVE else None,
    drive_root=Path(DRIVE_ROOT) / "gallery-export" if EXPORT_TO_DRIVE else None,
    download=DOWNLOAD_ZIP,
)
close_bars()

if "zip" in outputs:
    print(f"zip    : {outputs['zip']['path']}")
    print(f"size   : {outputs['zip']['bytes']} bytes")
    print(f"sha256 : {outputs['zip']['sha256']}")
if "drive" in outputs:
    print(f"drive  : {outputs['drive']['path']}")
print(f"report : {outputs['run_report']}")

# The runtime record travels with the gallery, so a rebuild on a different image
# can be compared with this run after /content has been recycled.
if RUNTIME_SNAPSHOT.is_file():
    shutil.copy(RUNTIME_SNAPSHOT, CONFIG.state_dir / "runtime.json")
    print(f"runtime  : {CONFIG.state_dir / 'runtime.json'}")

In [ ]:
# @title 12b · Run report — keep this file next to the dataset
"""The last block is the point of this cell: it prints what a future rebuild needs
in order to agree with this run — or to say honestly that it cannot, because a
checkout was never verified against a pin."""
import json

report_path = RUNNER.config.state_dir / "run_report.json"
report = json.loads(report_path.read_text())

print(json.dumps({key: report[key] for key in ("created_at", "gpu", "embedders")}, indent=2))
print("\nsummary:")
for key, value in report["summary"].items():
    print(f"  {key:<10} {value}")
print("\nstages:")
for stage in report["stages"]:
    print(
        f"  {stage['name']:<9} processed={stage['processed']:<6}"
        f" skipped={stage['skipped']:<6} failed={stage['failed']:<4}"
        f" {stage['seconds']:>7.1f}s"
    )

print("\nreproducibility:")
source = report.get("source") or {}
if source:
    revision = (source.get("revision") or "")[:12]
    state = "DIRTY tree" if source.get("dirty") else "clean tree"
    print(f"  code        {source.get('repo')}@{revision} ({state}, {source.get('action')})")
else:
    print("  code        NOT RECORDED — this run did not verify a checkout against a pin")

environment = report.get("environment") or {}
spec = environment.get("spec") or {}
drift = environment.get("drift") or {}
packages = environment.get("packages") or {}
spec_sha = (spec.get("sha256") or "unhashed")[:12]
print(
    f"  environment {spec_sha}, torch {packages.get('torch') or 'absent'},"
    f" {len(drift)} package(s) differ from the recorded baseline"
)

checkpoints = report.get("checkpoints") or {}
lock = checkpoints.get("lock") or {}
artifacts = checkpoints.get("artifacts") or []
lock_sha = (lock.get("sha256") or "unhashed")[:12]
print(
    f"  checkpoints {lock_sha} at policy {checkpoints.get('policy')},"
    f" {len(artifacts)} artifact(s) touched this run"
)
for item in artifacts:
    print(f"    {item.get('status'):<9} {item.get('id')} {(item.get('sha256') or '')[:12]}")
if not artifacts:
    print("    (none yet: the GPU stages record weights as they load them)")

### Hands-off alternative

Prefer one cell? `run_all` chains the same eight stages in the same order and
returns the run report — it is a convenience, not a second pipeline. It needs cells
1–4 first, because `CONFIG` is built from what cells 2 and 2d measured: the checkout
verdict, the resolved sources, the environment digest. Use it after you have watched
the stages run once, so you know what the numbers should look like.

```python
runner = PipelineRunner(CONFIG, progress=_progress, on_note=_note)
report = runner.run_all(
    zip_path=ZIP_PATH if DOWNLOAD_ZIP else None,
    drive_root=Path(DRIVE_ROOT) / "gallery-export" if EXPORT_TO_DRIVE else None,
    download=DOWNLOAD_ZIP,
)
# Cell 12 copies the runtime record next to the manifest; run_all does not, and a
# gallery missing that one file is a gallery you cannot compare against later.
if RUNTIME_SNAPSHOT.is_file():
    import shutil

    shutil.copy(RUNTIME_SNAPSHOT, CONFIG.state_dir / "runtime.json")
```

The reproducibility fields are already in `CONFIG`, so a one-cell run records the same
commit, environment digest, and checkpoint digests in `run_report.json` and the
manifest — the copy above is the only artifact the two paths would otherwise disagree
about.

## 13 · Back on your machine

1. Unzip the gallery into the repository's ignored data directory:
   ```bash
   unzip linescout-<version>.zip -d data/gallery/<version>/
   # feature shards (Milestone 4 index building) go to:
   #   data/indexes/<version>/{mobileclip2_s2,dinov2_vits14}/
   ```
2. Point the API at it and turn on curation — in `services/api/.env`:
   ```bash
   LINESCOUT_GALLERY_MANIFEST=data/gallery/<version>/manifest.json
   LINESCOUT_CURATION_MODE=1
   ```
3. Validate, then boot:
   ```bash
   npm run setup:py
   ml/.venv/bin/linescout-manifest validate data/gallery/<version>/manifest.json --require-files
   npm run dev:all          # API on :8000, web on :5173
   ```
4. Curate at `http://127.0.0.1:5173/curate`. Every asset arrives
   `review.state="unreviewed"` with `enabled=false`. Curation preview routes
   let reviewers see it without exposing unreviewed assets on search.
5. When the batch is done, `POST /api/v1/curation/snapshots` writes an immutable
   label snapshot — the artefact a later manifest re-export patches from.

**Nothing here needs to run twice.** Candidate state, manifest merge, and the
embedding index are all idempotent: re-run any cell and it picks up where it
stopped.

## 14 · Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `nvidia-smi not found` | CPU runtime | Runtime → Change runtime type → T4 GPU. Everything still runs; extraction is ~30x slower |
| `CUDA out of memory` | one huge scan | the runner already retries that image at half resolution; lower `LINE_ART_RESOLUTION` or `BATCH_SIZE` if it recurs |
| `RuntimeError: … disconnected` | Colab recycled the VM | re-run cells 2, 2b, 2c, 2d, 4, 4b, then the stage you were in — `candidates.jsonl` and the embedding index make it resume |
| `the pinned checkout is not usable` | `REPO_PIN` names a commit the remote does not have | the commit is unpublished, or lives in another repository: `ml/colab/README.md` names the current pin, and `REPO_URL` must be a repository that carries it |
| `REPO_DIR=… is not a LineScout checkout` | that folder is not the repo | leave `REPO_DIR` blank to clone, or point it at the real checkout (it must contain `ml/linescout_ml`) |
| `Refusing to move a checkout with uncommitted edits` | your Drive copy is dirty at the wrong revision | commit or stash there, set `REPO_DIR` elsewhere, or delete that copy — the notebook will not reset your work for you |
| `refuse a dirty tree` | `REPO_ALLOW_DIRTY = False` and HEAD is right but edited | commit, or set `REPO_ALLOW_DIRTY = True` and accept a run recorded as dirty |
| `pip refused <package>` | the pinned environment does not fit this image | do **not** install it unpinned: the run stops being comparable. Keep `linescout-runtime.json`, and file it with the report |
| `MissingDependencyError` | a stage's package is absent | cell 2d installs what cell 2b resolved; enable the toggle in cell 1 and re-run both |
| `does not match the pinned SHA-256` | a cached weight file differs from `models.lock.json` | the pinned digest is the reviewed one. Clear `CHECKPOINT_CACHE_DIR` and re-download; if the upstream file genuinely changed, pin the new bytes with `linescout-repro checkpoints record` and open a PR |
| `source 'x' needs a real license_id` | placeholder licence | intentional — put the licence you were actually granted in `SOURCES` (cell 2b stops the run before any download) |
| `FileNotFoundError: … root does not exist` | Drive path typo | check `SOURCES_ROOT`; Drive mounts at `/content/drive/MyDrive/…` |
| `gallery is incomplete` | a file vanished mid-run | re-run cell 6; missing outputs are rewritten, existing ones skipped |
| `queue_empty` in the curation UI | everything already reviewed | run `scripts/reset_reviews.py` against a copy of the database |
| Line art looks like grey mush | wrong extractor for the source | `anime2sketch` for manga/anime, `informative_drawings` for photos and paintings; set `extractor` per source in `SOURCES` |

**Free-tier reality check.** Colab disconnects idle and heavy sessions, so the
pipeline is built for that: every stage persists before the next one starts,
embeddings are sharded, and the manifest is merged rather than rewritten. Treat a
run as a series of resumable sessions, not one long transaction.

**Checking pins from your own machine.** Everything these cells do around the pins
is also a command, and it needs no GPU:

```bash
cd ml && uv sync --frozen --extra dev
.venv/bin/linescout-repro environment plan --preset human_art
.venv/bin/linescout-repro checkpoints show
.venv/bin/linescout-repro checkout --dir ../drawable-check
.venv/bin/linescout-repro selfcheck     # notebook ↔ package ↔ pins ↔ docs audit
```

## 15 · Attribution and licences

The manifest records a licence for every **asset**. This table is about the
**models** whose weights ran to produce them. Both matter, and only one of them
is comfortable.

| Component | Used for | Licence | Note |
|---|---|---|---|
| [Anime2Sketch](https://github.com/Mukosame/Anime2Sketch) (`netG.pth`) | line art from manga/anime | MIT | weights mirrored on HF as `lllyasviel/Annotators` |
| [Informative Drawings](https://github.com/carolineec/informative-drawings) (`sk_model.pth`) | line art from photos/paintings | MIT | Chan *et al.*, CVPR 2022 |
| [controlnet_aux](https://github.com/huggingface/controlnet_aux) | loads both of the above | Apache-2.0 | why no Drive checkpoint hunting is needed |
| [MobileCLIP2](https://github.com/apple/ml-mobileclip) | zero-shot labels + 512-d features | code MIT, **weights Apple ML Research Model License** | **research purposes only, no commercial use, no commercial products.** Features derived from it inherit that restriction — record it in your model card |
| [DINOv2](https://github.com/facebookresearch/dinov2) | 384-d shape features | Apache-2.0 (code and weights) | the specialised XRay/Cell variants are *not* Apache; this pipeline never loads them |
| [opennsfw2](https://github.com/bhaveshgohel/opennsfw2) | optional SFW screen | MIT | needs TensorFlow; Colab ships it |

Cite, if this dataset supports published work:

* Faghri *et al.*, **MobileCLIP2: Improving Multi-Modal Reinforced Training**, TMLR 2025
* Oquab *et al.*, **DINOv2: Learning Robust Visual Features without Supervision**, 2023
* Chan *et al.*, **Informative Drawings**, CVPR 2022
* Zhu *et al.*, **Anime2Sketch: A Sketch Extractor for Anime Arts with Deep Networks**, 2021

**Dataset licences are yours to verify.** Cell 4 prints what to check for each
preset and deliberately ships no `license_id`: a provenance manifest that guesses
at licences is worse than no manifest at all.